# Do-as-I-Do · Reconstruction — RunPod

End-to-end hand + object **reconstruction and 6-DoF pose tracking** from a single demo video,
running the [`reconstruction/`](https://github.com/malik-group/do-as-i-do) pipeline on a RunPod
**A100 80 GB** GPU pod, from a Jupyter notebook.

### RunPod prerequisites (do these in the RunPod dashboard / pod first)
1. **Pod template.** Launch an **NVIDIA A100 80 GB** pod with a CUDA 12.x PyTorch template
   (e.g. RunPod's official `PyTorch` template). The CUDA toolkit (`nvcc` at `/usr/local/cuda`) is
   used to compile a few extensions (pytorch3d, DROID-SLAM), so prefer a dev/PyTorch image over a
   bare runtime image.
2. **Network volume (recommended).** Attach a network volume at `/workspace` so the cloned repo,
   weights, video, and MANO files persist across pod restarts. Otherwise everything lives on the
   container disk and is lost when the pod stops.
3. **Upload your inputs** onto the pod (via the RunPod file browser or `scp`/`rclone`):
   - your demo **video** (e.g. `/workspace/whisking.mp4`);
   - `MANO_RIGHT.pkl` and `MANO_LEFT.pkl` (manual, license-gated download from
     https://mano.is.tue.mpg.de) into a folder, e.g. `/workspace/mano/`.
4. **HuggingFace access.** Request access to the gated repos `facebook/sam-3d-objects` and
   `facebook/sam3`. Have a token ready (https://huggingface.co/settings/tokens) — you'll paste it
   in the auth cell, or export it as `HF_TOKEN` before launching Jupyter.

### What this notebook does
Installs Miniconda, builds the pipeline's **4 conda envs** (`sam3`, `sam3d`, `hawor`, `tapnet`),
fetches weights, lets you **click the object** on the reference frame (interactive matplotlib
`ginput` — works in real Jupyter, unlike Colab), and runs `run_pipeline.sh` end-to-end with the
click substituted by your points.

Run cells top-to-bottom. **[setup]** cells run once per pod; **[run]** cells are per-video.

## 0 · Configuration  [run]

All paths are **local filesystem paths on the pod** (no Google Drive). Edit for your video.

In [ ]:
import os

# --- Your video (path on the pod) ---
VIDEO_PATH = "/workspace/pipette.mp4"  # edit me

# --- Reference frame + object + anchor hand (same args as run_pipeline.sh) ---
FRAME_N = 125   # edit me
OBJECT  = "pipette"   # edit me
ANCHOR_HAND = "right"   # "right" or "left"

# --- MANO models (folder containing MANO_RIGHT.pkl + MANO_LEFT.pkl) ---
MANO_DIR = "/workspace/mano"    # edit me

# --- Where to clone the repo (put it on the network volume for persistence) ---
REPO_DIR = "/workspace/do-as-i-do"

# Expose to subsequent %%bash cells.
for k, v in {"VIDEO_PATH": VIDEO_PATH, "FRAME_N": str(FRAME_N), "OBJECT": OBJECT,
             "ANCHOR_HAND": ANCHOR_HAND, "MANO_DIR": MANO_DIR, "REPO_DIR": REPO_DIR}.items():
    os.environ[k] = v
print("VIDEO_PATH =", VIDEO_PATH)
print("REPO_DIR   =", REPO_DIR)

## 1 · GPU & disk sanity check  [setup]

Asserts the pod has an A100 (≥ 32 GB VRAM) and enough local disk for envs + weights (~40 GB).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
VRAM_MB=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1 | tr -d ' ')
if [ "$VRAM_MB" -lt 30000 ]; then
  echo "!! This GPU has only ${VRAM_MB} MB VRAM. The pipeline needs >= 32 GB."
fi
echo "--- nvcc (needed to compile extensions) ---"
nvcc --version 2>/dev/null || ls /usr/local/cuda/bin/nvcc 2>/dev/null || echo "nvcc not found — install a CUDA dev image"
echo "--- disk ---"
df -h /workspace 2>/dev/null || df -h /

## 2 · Notebook-kernel deps + HuggingFace login  [setup]

These packages are installed into the **Jupyter kernel's Python** (not the conda envs) — needed
for the interactive click plot later. Then authenticate to HuggingFace so the gated
`facebook/sam-3d-objects` / `facebook/sam3` checkpoints can be downloaded.

In [ ]:
# Kernel-side deps for the interactive click cell.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl", "matplotlib", "opencv-python", "numpy"])
print("kernel deps installed.")

In [ ]:
import os, getpass
# Prefer an existing HF_TOKEN env var; otherwise prompt (kept out of cell history via getpass).
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your HuggingFace token (input hidden): ")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF token set (length %d)." % len(os.environ["HF_TOKEN"]))

In [ ]:
%%bash
pip install -q 'huggingface-hub[cli]<1.0'
git config --global credential.helper store
hf auth login --token "$HF_TOKEN" --add-to-git-credential

## 3 · Install Miniconda  [setup]

If the pod image already has conda at `/opt/conda`, this is a no-op; otherwise it installs
Miniconda there. Every later `%%bash` cell re-sources it (shell state does not persist between
cells).

In [ ]:
%%bash
set -e
if [ -x /opt/conda/bin/conda ]; then
  echo "conda already installed at /opt/conda"
else
  echo "Installing Miniconda..."
  cd /tmp
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
  bash miniconda.sh -bfp /opt/conda
  rm miniconda.sh
fi
source /opt/conda/etc/profile.d/conda.sh
conda --version
# Recent conda refuses env creation until the defaults channels' ToS are accepted.
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true

## 4 · Clone the repo + submodules  [setup]

`GIT_LFS_SKIP_SMUDGE=1` so the heavy weight blobs are **not** pulled by Git LFS — they come from
`setup/02_fetch_weights.sh` later.

In [ ]:
%%bash
set -e
if [ -d "$REPO_DIR/.git" ]; then
  echo "Repo already cloned at $REPO_DIR"
else
  mkdir -p "$(dirname "$REPO_DIR")"
  cd "$(dirname "$REPO_DIR")"
  GIT_LFS_SKIP_SMUDGE=1 git clone --recurse-submodules https://github.com/malik-group/do-as-i-do.git "$(basename "$REPO_DIR")"
fi
cd "$REPO_DIR"
GIT_LFS_SKIP_SMUDGE=1 git submodule update --init --recursive
echo "--- submodule pins ---"
git submodule status

## 5 · Build the 4 conda envs  [setup]

Each cell runs the repo's own `setup/01_create_envs.sh <env>`. These are slow (~5-10 min each)
and use a lot of disk; run once.

- **`sam3`** and **`tapnet`** run unchanged (their recipes already install cu128 torch, fine on A100).
- **`sam3d`** needs two env vars set first (`PIP_EXTRA_INDEX_URL` + `PIP_FIND_LINKS`) so pip can
  find the `+cu121` torch wheels and the kaolin wheel bucket — `01_create_envs.sh` omits these,
  which is why a bare run fails on `torchaudio==2.5.1+cu121`. (Mirrors `sam-3d-objects/doc/setup.md`.)
  `CUDA_HOME`/`TORCH_CUDA_ARCH_LIST` are exported so any source-built extension targets sm_80.
- **`hawor`** runs unchanged (cu117/torch 1.13 runs fine on A100 sm_80).

If a cell fails, just re-run **that** cell — `conda env create` falls back to `env update`.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/01_create_envs.sh sam3

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"

# sam-3d-objects/requirements.txt pins torch 2.5.1 / torchvision / torchaudio with the +cu121
# local version, and kaolin comes from NVIDIA's wheel bucket. setup/01_create_envs.sh doesn't
# set these, so pip can't find torchaudio==2.5.1+cu121. Mirrors sam-3d-objects/doc/setup.md.
export PIP_EXTRA_INDEX_URL="https://pypi.ngc.nvidia.com https://download.pytorch.org/whl/cu121"
export PIP_FIND_LINKS="https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html"
# If any extension compiles from source, target A100 (sm_80) and use the pod's CUDA toolkit:
export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
export TORCH_CUDA_ARCH_LIST="8.0"
bash setup/01_create_envs.sh sam3d

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
RECON="$REPO_DIR/reconstruction"
HAWOR_DIR="$RECON/modules/HaWoR"
cd "$RECON"

# Replicate setup/01_create_envs.sh (mk_hawor) inline, with two fixes this pod needs:
#  (1) pytorch3d's setup.py does `import torch` at the top, so it MUST build with
#      --no-build-isolation against the torch installed below — otherwise PEP 517 build
#      isolation hides torch and fails with `ModuleNotFoundError: No module named 'torch'`.
#  (2) compile its CUDA kernels with the pod's toolkit, targeting A100 (sm_80).
if ! conda env list | grep -q '^hawor '; then
  conda create -y -n hawor python=3.10
fi
conda activate hawor
conda install -y -c conda-forge ffmpeg

# torch (cu117 per HaWoR README; runs fine on A100 sm_80)
pip install torch==1.13.0+cu117 torchvision==0.14.0+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117
pip install ninja setuptools wheel          # build deps for the pytorch3d compile

# Everything in requirements.txt EXCEPT mmcv (won't build on modern torch), chumpy@git,
# and pytorch3d (handled next — needs --no-build-isolation).
grep -viE "mmcv==1.3.9|chumpy@|pytorch3d" "$HAWOR_DIR/requirements.txt" \
  | pip install -r /dev/stdin

# pytorch3d: build against the installed torch (no isolation), targeting sm_80.
P3D=$(grep -iE "pytorch3d" "$HAWOR_DIR/requirements.txt")
export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
export TORCH_CUDA_ARCH_LIST="8.0"
export FORCE_CUDA=1
pip install --no-build-isolation "$P3D"

# chumpy 0.71 is git-only (PyPI max 0.70); needs numpy at build time.
pip install "chumpy@git+https://github.com/mattloper/chumpy" --no-build-isolation
pip install "setuptools<81"                   # pytorch-lightning 2.2.4 needs pkg_resources
pip install pytorch-lightning==2.2.4 --no-deps
pip install lightning-utilities torchmetrics==1.4.0

# DROID-SLAM (lietorch + droid-backends). No dispatch.h patch needed on torch 1.13.
( cd "$HAWOR_DIR/thirdparty/DROID-SLAM" && python setup.py install )

# torch>=2.6 loads checkpoints weights_only=True, which rejects HaWoR's omegaconf-bearing
# ckpts. Restore the pre-2.6 default for this env.
conda env config vars set TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 -n hawor
echo "=== hawor env ready ==="
python -c "import torch,pytorch3d; print('torch',torch.__version__,'cuda',torch.version.cuda)"

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/01_create_envs.sh tapnet

## 6 · `sam3d` post-install fixes  [setup]

Two manual steps the README calls out after building the `sam3d` env
([`env/README.md`](https://github.com/malik-group/do-as-i-do/blob/main/reconstruction/env/README.md)):

1. `pip uninstall -y notebook` so `notebook.inference` (vendored inside sam-3d-objects) imports.
2. Build the Mip-Splatting `diff_gaussian_rasterization` (the `inria` GLB/texture baking backend),
   only if it isn't already importable.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3d

# 1. un-shadow the repo's notebook/ package
pip uninstall -y notebook 2>/dev/null || true

# 2. Mip-Splatting gaussian rasterizer (only if not already importable)
python - <<'PY'
import importlib.util as u
if u.find_spec("diff_gaussian_rasterization") is None:
    import subprocess, os
    work = "/workspace/mip-splatting-build"
    if not os.path.isdir(work):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/autonomousvision/mip-splatting.git", work], check=True)
    sub = os.path.join(work, "submodules", "diff-gaussian-rasterization")
    env = os.environ.copy()
    env["CUDA_HOME"] = os.environ.get("CUDA_HOME") or "/usr/local/cuda"
    env["TORCH_CUDA_ARCH_LIST"] = "8.0"     # A100 (sm_80)
    env["FORCE_CUDA"] = "1"
    subprocess.run(["python", "setup.py", "install"], cwd=sub, env=env, check=True)
    print("diff_gaussian_rasterization built.")
else:
    print("diff_gaussian_rasterization already installed; skipping.")
PY

## 7 · Fetch model weights  [setup]

Runs the repo's `setup/02_fetch_weights.sh --download`. Pulls the **SAM3D** checkpoint set from
HuggingFace (gated), the **HaWoR** / **Metric3D** / **DROID-SLAM** checkpoints, and the
**BootsTAPIR** checkpoint, then symlinks the shared heavy SAM3D files into both module dirs.

The Stage-1 **SAM3** model is auto-downloaded by `run_sam3_video.py` at runtime (also gated).

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/02_fetch_weights.sh --download

## 8 · Place MANO hand models  [setup]

MANO is license-gated and cannot be auto-downloaded. Copy the two `.pkl` files you downloaded
from https://mano.is.tue.mpg.de (uploaded to the pod at `MANO_DIR`) into the paths HaWoR expects.

In [ ]:
import os, shutil, sys

HAWOR = os.path.join(os.environ["REPO_DIR"], "reconstruction/modules/HaWoR")
targets = {
    "MANO_RIGHT.pkl": f"{HAWOR}/_DATA/data/mano/MANO_RIGHT.pkl",
    "MANO_LEFT.pkl":  f"{HAWOR}/_DATA/data_left/mano_left/MANO_LEFT.pkl",
}
missing = []
for name, dst in targets.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    src = os.path.join(os.environ["MANO_DIR"], name)
    if os.path.isfile(src):
        shutil.copy2(src, dst); print(f"placed {name} -> {dst}")
    elif os.path.isfile(dst):
        print(f"{name} already present at {dst}")
    else:
        missing.append(src)
if missing:
    print("!! Could not find these MANO files (expected in MANO_DIR):")
    for m in missing: print("   ", m)
    sys.exit(1)
print("MANO models in place.")

## 9 · Setup sanity check  [setup]

Confirms all 4 envs exist and the key weights are where the pipeline expects them.

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
echo "=== conda envs ==="
conda env list
echo; echo "=== torch in each env ==="
for e in sam3 sam3d hawor tapnet; do
  echo -n "$e: "
  conda run -n "$e" python -c "import torch; print(torch.__version__, 'cuda=' + str(torch.version.cuda), 'avail=' + str(torch.cuda.is_available()))" 2>/dev/null || echo "(torch import failed)"
done
echo; echo "=== weight presence ==="
R="$REPO_DIR/reconstruction"
ls -lh "$R/weights/tapnet/bootstapir_checkpoint_v2.pt" 2>/dev/null || echo "MISSING: tapnet ckpt"
ls -lh "$R/weights/sam3d_shared/hf/" 2>/dev/null | head
ls -lh "$R/modules/HaWoR/weights/hawor/checkpoints/hawor.ckpt" 2>/dev/null || echo "MISSING: hawor.ckpt"
ls -lh "$R/modules/HaWoR/_DATA/data/mano/MANO_RIGHT.pkl" 2>/dev/null || echo "MISSING: MANO_RIGHT.pkl"
echo; echo "Setup complete if nothing above says MISSING / failed."
echo "Then continue to the Run section."

---
# Run phase (per-video)

Cells below are **[run]** — re-run them whenever you change `VIDEO_PATH` / `FRAME_N` / etc.

## 10 · Locate the video  [run]

The video already lives on the pod (no Drive to copy from). The pipeline writes its outputs
*next to* the video, so make sure `VIDEO_PATH` is on writable storage (network volume or
container disk).

In [ ]:
import os
VIDEO_PATH = os.environ["VIDEO_PATH"]
assert os.path.isfile(VIDEO_PATH), f"video not found: {VIDEO_PATH}"
VIDEO_DIR  = os.path.dirname(VIDEO_PATH)
os.environ["VIDEO_LOCAL"] = VIDEO_PATH
os.environ["VIDEO_DIR"]   = VIDEO_DIR
print("video:", VIDEO_PATH)
print("outputs will be written under:", VIDEO_DIR)

## 11 · Extract the reference frame for clicking  [run]

Uses the exact same `ffmpeg` invocation `run_pipeline.sh` will use in Step 0, so the frame
numbering matches the tracking output.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3   # provides ffmpeg (same as run_pipeline.sh Step 0)
mkdir -p "$VIDEO_DIR/all_frames"
# Same flags as run_pipeline.sh Step 0 (numbering starts at 0)
ffmpeg -y -i "$VIDEO_LOCAL" -vsync 0 -start_number 0 "$VIDEO_DIR/all_frames/%06d.png"
# Pull out the single reference frame for the click UI.
REF="$VIDEO_DIR/$(printf '%04d.png' "$FRAME_N")"
ffmpeg -y -i "$VIDEO_LOCAL" -vf "select=eq(n\,$FRAME_N)" -vsync 0 -vframes 1 "$REF"
echo "reference frame: $REF"

## 12 · Click the object on the reference frame  [run]

RunPod runs a real Jupyter kernel, so we use matplotlib `ginput` — click directly on the image.
**Requires the `ipympl` backend** (`%matplotlib widget`, installed in §2). Click 1-3 points on
the object, then **press Enter** in the figure (or close it) to finish.

If the figure isn't interactive (no zoom/pan toolbar), make sure the notebook is running under
JupyterLab/classic with `ipympl` installed, and that you selected the `widget` matplotlib backend.

In [ ]:
%matplotlib widget
import os, cv2
import matplotlib.pyplot as plt

FRAME_N = int(os.environ["FRAME_N"])
REF_PNG = os.path.join(os.environ["VIDEO_DIR"], f"{FRAME_N:04d}.png")
img = cv2.cvtColor(cv2.imread(REF_PNG), cv2.COLOR_BGR2RGB)
assert img is not None, f"could not read {REF_PNG}"

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img); ax.axis('off')
ax.set_title(f"Click on the '{os.environ['OBJECT']}' (any number of points).\nPress Enter when done.")
clicks = plt.ginput(-1, timeout=-1, show_clicks=True)   # -1 = until Enter / close
plt.close(fig)
clicks = [(int(round(x)), int(round(y))) for x, y in clicks]
assert clicks, "no points collected — re-run this cell and click the object"
print("collected points:", clicks)

In [ ]:
# Format the collected clicks into the --points / --point_labels args run_sam3_video.py expects.
import os
OBJ_POINTS       = ";".join(f"{x},{y}" for x, y in clicks)
OBJ_POINT_LABELS = ";".join("1" for _ in clicks)   # all positive
print("Object points :", OBJ_POINTS)
print("Labels        :", OBJ_POINT_LABELS)
os.environ["OBJ_POINTS"] = OBJ_POINTS
os.environ["OBJ_POINT_LABELS"] = OBJ_POINT_LABELS

## 13 · Run the full pipeline  [run]

We make a one-line patch to `run_pipeline.sh`: swap the single `--click` token for a `--points`
/ `--point_labels` pair fed by the clicks you collected above. Everything else (env switching,
frame extraction, all stages 0-4) is the repo's own driver, untouched.

In [ ]:
# Patch a copy of run_pipeline.sh: swap `--click` for a --points/--point_labels pair, so Stage 1
# runs headlessly using the clicks you just collected. The trailing backslash that followed
# `--click` in the original is left in place, so line continuation still works.
import os
recon = os.path.join(os.environ["REPO_DIR"], "reconstruction")
src = open(f"{recon}/run_pipeline.sh").read()
assert src.count("--click") == 1, f"expected exactly one --click in run_pipeline.sh, found {src.count('--click')}"
repl = '--points "$OBJ_POINTS" --point_labels "$OBJ_POINT_LABELS"'
open(f"{recon}/run_pipeline_colab.sh", "w").write(src.replace("--click", repl))
print("wrote run_pipeline_colab.sh")

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
chmod +x run_pipeline_colab.sh
export OBJ_POINTS="$OBJ_POINTS"
export OBJ_POINT_LABELS="$OBJ_POINT_LABELS"
echo "=== launching patched pipeline ==="
./run_pipeline_colab.sh "$VIDEO_LOCAL" "$FRAME_N" "$OBJECT" "$ANCHOR_HAND"

## 14 · Done — where the outputs live  [run]

All outputs are written next to the video under `$VIDEO_DIR/`. The directory consumed by the
[`retargeting/`](https://github.com/malik-group/do-as-i-do/tree/main/retargeting) pipeline is:

```
<VIDEO_DIR>/obj_tracking_out/<OBJECT>/combined_visualization/
    layout_camera_frame_optimized.json      <- final 6-DoF object pose track
    projected_*.png                          <- projected-mesh overlays
```

Also produced: per-object `.obj` meshes in `video_segmentation/masks/frame_NNNNNN_masks/<OBJECT>/`,
HaWoR `all_hand_meshes.npz`, per-frame `*_pointmap.npy` / `*_intrinsics.*`, and `gravity.json`.

In [ ]:
%%bash
VIDEO_DIR="$(dirname "$VIDEO_LOCAL")"
echo "=== tree of outputs ==="
find "$VIDEO_DIR" -maxdepth 3 -type d | sort
echo; echo "=== final layout json ==="
ls -la "$VIDEO_DIR/obj_tracking_out/$OBJECT/combined_visualization/layout_camera_frame_optimized.json" 2>/dev/null \
  && echo "OK: reconstruction finished successfully" \
  || echo "!! optimized layout json missing — check the pipeline log above."